[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_dnam.ipynb) [![Open In nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_dnam.ipynb)

# DNA methylation

Run clocks on human arrays, mammalian arrays, or mouse RRBS data.

- [Human arrays](#human-arrays)
- [Mammalian arrays](#mammalian-arrays)
- [Mouse RRBS](#mouse-rrbs)

We just need two packages for this tutorial.

In [1]:
import pandas as pd
import pyaging as pya

<a id="human-arrays"></a>

## Human arrays

### Download and load example data

Let's download the publicly avaiable dataset GSE139307 with Illumina's 450k array. The CpG coverage of the 450k array should be good enough for most clocks.

In [2]:
pya.data.download_example_data('GSE139307')

⏺ example data already at pyaging_data/GSE139307.pkl

'pyaging_data/GSE139307.pkl'

In [3]:
df = pd.read_pickle('pyaging_data/GSE139307.pkl')

In [4]:
df.head()

,dataset,tissue_type,age,gender,cg00000029,cg00000108,cg00000109,cg00000165,cg00000236,cg00000289,...,ch.X.93511680F,ch.X.938089F,ch.X.94051109R,ch.X.94260649R,ch.X.967194F,ch.X.97129969R,ch.X.97133160R,ch.X.97651759F,ch.X.97737721F,ch.X.98007042R
GSM4137709,GSE139307,sperm,84.0,M,0.084811,0.920696,0.856851,0.084567,0.838699,0.247273,...,0.061751,0.045942,0.037631,0.056455,0.249872,0.049022,0.085691,0.037435,0.077820,0.106234
GSM4137710,GSE139307,sperm,69.0,M,0.099626,0.919073,0.890024,0.115541,0.852584,0.198103,...,0.075077,0.041849,0.032573,0.089790,0.250245,0.079095,0.079756,0.046229,0.091256,0.120241
GSM4137711,GSE139307,sperm,69.0,M,0.117228,0.920276,0.894317,0.117127,0.839258,0.213410,...,0.068679,0.049515,0.058097,0.079919,0.299758,0.079305,0.089815,0.065364,0.086864,0.156005
GSM4137712,GSE139307,sperm,69.0,M,0.077096,0.910204,0.908400,0.073885,0.861615,0.163276,...,0.070091,0.033289,0.038836,0.108213,0.295428,0.050731,0.099943,0.047597,0.078480,0.107480
GSM4137713,GSE139307,sperm,67.0,M,0.063524,0.911608,0.884643,0.079877,0.864654,0.176169,...,0.082368,0.038411,0.048787,0.088631,0.316694,0.041873,0.079303,0.048823,0.089010,0.117903


For PCGrimAge and GrimAge2, both age and sex are features. Therefore, to get the full prediction, let's convert the column `gender` into a column called `female`, with 1 being female and 0 being male.

In [5]:
# needs only numerical data (doesn't work with strings)
df['female'] = (df['gender'] == 'F').astype(int)

Moreover, it is important to note that some probes are duplicated in the EPICv2 array, following the format cg#########_BC11 and cg#########_TC11 for the opposite strands. Given that at this moment most clocks have not been trained with EPICv2 data directly, it is recommended to average these probes. This is particularly the case for DunedinPACE, from which some clock probes were duplicated in the update from EPICv1. To remedy this issue, simply use the following function to aggregate any duplicated probes that may be present.

In [6]:
df = pya.pp.epicv2_probe_aggregation(df)

### Convert data to AnnData object

AnnData objects are highly flexible and are thus our preferred method of organizing data for age prediction.

In [7]:
adata = pya.pp.df_to_adata(df, metadata_cols=['gender', 'tissue_type', 'dataset'], imputer_strategy='knn')

Note that the original DataFrame is stored in `X_original` under layers. is This is what the `adata` object looks like:

In [8]:
adata

AnnData object with n_obs × n_vars = 37 × 485513
    obs: 'gender', 'tissue_type', 'dataset'
    var: 'percent_na'
    uns: 'imputer_strategy'
    layers: 'X_original', None (.X), 'X_imputed'

### Predict age

We can either predict one clock at once or all at the same time. For convenience, let's simply input all four clocks of interest at once. The function is invariant to the capitalization of the clock name. 

In [ ]:
pya.pred.predict_age(adata, ['horvath2013', 'altumage', 'pcgrimage', 'grimage2', 'dunedinpace'])

In [10]:
adata.obs.head()

,gender,tissue_type,dataset,horvath2013,altumage,pcgrimage,grimage2,dunedinpace
GSM4137709,M,sperm,GSE139307,33.624776,37.007213,95.506114,77.581057,1.326327
GSM4137710,M,sperm,GSE139307,28.829344,29.426899,83.934244,65.926346,1.215611
GSM4137711,M,sperm,GSE139307,28.316545,22.798928,82.709334,63.358341,1.271091
GSM4137712,M,sperm,GSE139307,24.850630,18.079173,84.269462,60.218880,1.276866
GSM4137713,M,sperm,GSE139307,25.942111,20.071985,84.356985,61.235919,1.262023


For curiosity, we can also check if there are any correlations amongst these clocks.

In [11]:
adata.obs.iloc[:, 3:].corr('pearson')

,horvath2013,altumage,pcgrimage,grimage2,dunedinpace
horvath2013,1.000000,0.676242,0.211881,0.459193,0.354771
altumage,0.676242,1.000000,0.156456,0.440044,0.164102
pcgrimage,0.211881,0.156456,1.000000,0.859490,0.061486
grimage2,0.459193,0.440044,0.859490,1.000000,0.183721
dunedinpace,0.354771,0.164102,0.061486,0.183721,1.000000


After age prediction, the clocks are added to `adata.obs`. Moreover, the percent of missing values for each clock and other metadata are included in `adata.uns`.

In [12]:
adata

AnnData object with n_obs × n_vars = 37 × 485513
    obs: 'gender', 'tissue_type', 'dataset', 'horvath2013', 'altumage', 'pcgrimage', 'grimage2', 'dunedinpace'
    var: 'percent_na'
    uns: 'imputer_strategy', 'horvath2013_percent_na', 'horvath2013_missing_features', 'horvath2013_metadata', 'altumage_percent_na', 'altumage_missing_features', 'altumage_metadata', 'pcgrimage_percent_na', 'pcgrimage_missing_features', 'pcgrimage_metadata', 'grimage2_percent_na', 'grimage2_missing_features', 'grimage2_metadata', 'dunedinpace_percent_na', 'dunedinpace_missing_features', 'dunedinpace_metadata'
    layers: 'X_original', None (.X), 'X_imputed'

We can also look at which features seem to be missing from each clock (if there are any).

In [13]:
adata.uns['dunedinpace_missing_features']

[]

### Get citation

The doi, citation, and some metadata are automatically added to the AnnData object under `adata.uns[CLOCKNAME_metadata]`.

In [14]:
adata.uns['horvath2013_metadata']

{'clock_name': 'horvath2013',
 'data_type': 'DNA methylation',
 'species': 'Homo sapiens',
 'year': 2013,
 'approved_by_author': '⌛',
 'citation': 'Horvath, S. DNA methylation age of human tissues and cell types. Genome Biology 14, R115 (2013).',
 'doi': 'https://doi.org/10.1186/gb-2013-14-10-r115',
 'notes': 'Pan-tissue DNAm-age predictor fitted by elastic net to a transformed chronological-age outcome and returned to the year scale; it uses 353 CpGs shared between the 27K and 450K arrays.',
 'research_only': None,
 'tissue': ['multi-tissue'],
 'predicts': ['chronological age'],
 'training_target': ['chronological age'],
 'unit': ['years'],
 'model_type': 'elastic net regression',
 'platform': ['Illumina 27K', 'Illumina 450K'],
 'population': 'all ages',
 'journal': 'Genome Biology',
 'last_author': 'Steve Horvath',
 'n_features': 353,
 'citations': 7318,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'postprocess': 'anti_log_linear',
 'reference_values': True}

In [15]:
adata.uns['altumage_metadata']

{'clock_name': 'altumage',
 'data_type': 'DNA methylation',
 'species': 'Homo sapiens',
 'year': 2022,
 'approved_by_author': '✅',
 'citation': 'de Lima Camillo, L.P., Lapierre, L.R. & Singh, R. A pan-tissue DNA-methylation epigenetic clock based on deep learning. npj Aging 8, 4 (2022).',
 'doi': 'https://doi.org/10.1038/s41514-022-00085-y',
 'notes': 'Pan-tissue chronological-age predictor using a five-hidden-layer neural network and 20,318 CpGs shared across the 27K, 450K and EPIC manifests; the actual training data came from 27K and 450K datasets.',
 'research_only': None,
 'tissue': ['multi-tissue'],
 'predicts': ['chronological age'],
 'training_target': ['chronological age'],
 'unit': ['years'],
 'model_type': 'deep neural network',
 'platform': ['Illumina 27K', 'Illumina 450K'],
 'population': 'all ages',
 'journal': 'npj Aging',
 'last_author': 'Ritambhara Singh',
 'n_features': 20318,
 'citations': 145,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'preprocess': 'sca

In [16]:
adata.uns['pcgrimage_metadata']

{'clock_name': 'pcgrimage',
 'data_type': 'DNA methylation',
 'species': 'Homo sapiens',
 'year': 2022,
 'approved_by_author': '⌛',
 'citation': 'Higgins-Chen, Albert T., et al. "A computational solution for bolstering reliability of epigenetic clocks: implications for clinical trials and longitudinal tracking." Nature Aging 2 (2022): 644–661.',
 'doi': 'https://doi.org/10.1038/s43587-022-00248-2',
 'notes': 'Principal-component proxy trained to reproduce the original DNAm GrimAge score; age and sex are additional model inputs.',
 'research_only': None,
 'tissue': ['whole blood'],
 'predicts': ['mortality risk'],
 'training_target': ['DNAm GrimAge output'],
 'unit': ['years'],
 'model_type': 'PCA + elastic net regression',
 'platform': ['Illumina 450K'],
 'population': 'adults',
 'journal': 'Nature Aging',
 'last_author': 'Morgan E. Levine',
 'n_features': 78466,
 'citations': 497,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'reference_values': True}

In [17]:
adata.uns['grimage2_metadata']

{'clock_name': 'grimage2',
 'data_type': 'DNA methylation',
 'species': 'Homo sapiens',
 'year': 2022,
 'approved_by_author': '⌛',
 'citation': 'Lu, A. T., et al. “DNA methylation GrimAge version 2.” Aging 14(23): 9484–9549 (2022).',
 'doi': 'https://doi.org/10.18632/aging.204434',
 'notes': 'Mortality-risk epigenetic clock combining ten blood DNAm surrogate biomarkers with chronological age and sex; the Cox linear predictor is calibrated to an age-like value in years.',
 'research_only': True,
 'tissue': ['whole blood'],
 'predicts': ['mortality risk'],
 'training_target': ['mortality'],
 'unit': ['years'],
 'model_type': 'elastic net Cox regression',
 'platform': ['Illumina 450K'],
 'population': 'older adults',
 'journal': 'Aging',
 'last_author': 'Steve Horvath',
 'n_features': 1032,
 'citations': 291,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'postprocess': 'cox_to_years',
 'reference_values': True}

In [18]:
adata.uns['dunedinpace_metadata']

{'clock_name': 'dunedinpace',
 'data_type': 'DNA methylation',
 'species': 'Homo sapiens',
 'year': 2022,
 'approved_by_author': '✅',
 'citation': 'Belsky, D. W., Caspi, A., Corcoran, D. L., et al. (2022). DunedinPACE, a DNA methylation biomarker of the pace of aging. eLife, 11, e73420.',
 'doi': 'https://doi.org/10.7554/elife.73420',
 'notes': 'Whole-blood elastic-net pace-of-aging biomarker trained at age 45 against a 20-year longitudinal slope composite of 19 organ-system biomarkers. PyAging follows the official 20,000-probe quantile-normalization panel: 173 scoring CpGs plus 19,827 background probes.',
 'research_only': True,
 'tissue': ['whole blood'],
 'predicts': ['pace of aging'],
 'training_target': ['pace of aging'],
 'unit': ['biological years per chronological year'],
 'model_type': 'elastic net regression',
 'platform': ['Illumina EPIC'],
 'population': 'adults',
 'journal': 'eLife',
 'last_author': 'Terrie E. Moffitt',
 'n_features': 20000,
 'citations': 967,
 'citations_

<a id="mammalian-arrays"></a>

## Mammalian arrays

In [1]:
import pandas as pd
import pyaging as pya

### Download and load example data

Let's download the publicly avaiable dataset GSE223748 with Illumina's Mammalian Methylation array. The CpG coverage of the this array (~37k) spans highly conserved CpG sequences. Let's download a subset of that data.

In [2]:
pya.data.download_example_data('GSE223748')

⏺ example data already at pyaging_data/GSE223748_subset.pkl

'pyaging_data/GSE223748_subset.pkl'

In [3]:
df = pd.read_pickle('pyaging_data/GSE223748_subset.pkl')

In [4]:
df.head()

,cg00000165,cg00001209,cg00001364,cg00001582,cg00002920,cg00003994,cg00004555,cg00005112,cg00005271,cg00006213,...,rs7746156_II_F_C_37550,rs798149_II_F_C_37528,rs845016_II_F_C_37529,rs877309_II_F_C_37552,rs9292570_I_F_C_37499,rs9363764_II_F_C_37541,rs939290_II_F_C_37535,rs951295_I_F_C_37507,rs966367_II_F_C_37551,rs9839873_II_F_C_37532
204509080002_R01C02,0.094879,0.916154,0.890314,0.053583,0.490381,0.034852,0.159705,0.763959,0.973245,0.928975,...,0.488592,0.491361,0.480024,0.500000,0.484252,0.489448,0.505585,0.505335,0.485003,0.510081
202897220142_R04C02,0.497077,0.441263,0.915314,0.047339,0.651029,0.037774,0.082634,0.415800,0.702857,0.821715,...,0.508102,0.500299,0.507261,0.490684,0.499673,0.497256,0.564106,0.482151,0.486667,0.505236
204529320092_R01C02,0.321141,0.834158,0.881194,0.056124,0.688350,0.030225,0.086776,0.777588,0.974587,0.923934,...,0.520404,0.509568,0.507549,0.501659,0.492823,0.487243,0.516018,0.471244,0.491066,0.491759
202794570004_R02C01,0.495226,0.924121,0.915812,0.050866,0.688335,0.032344,0.113318,0.872094,0.969189,0.917076,...,0.499314,0.516132,0.487009,0.487146,0.469119,0.495125,0.548238,0.512283,0.514257,0.492520
203531420070_R05C02,0.183954,0.934332,0.924153,0.055032,0.717495,0.037108,0.098632,0.859614,0.973422,0.963446,...,0.501432,0.509412,0.485055,0.497272,0.480637,0.467502,0.494246,0.500924,0.531334,0.503709


### Convert data to AnnData object

AnnData objects are highly flexible and are thus our preferred method of organizing data for age prediction.

In [5]:
adata = pya.pp.df_to_adata(df, imputer_strategy='knn')

This is what the `adata` object looks like:

In [6]:
adata

AnnData object with n_obs × n_vars = 100 × 37554
    var: 'percent_na'
    layers: 'X_original', None (.X)

### Predict age

#### Mammalian predictors without species declaration

We can either predict one clock at once or all at the same time. Let's first start with the mammalian clocks that do not need the species Latin name for the conversion of the output into units of years.

In [7]:
pya.pred.predict_age(adata, ['Mammalian1', 'MammalianLifespan', 'MammalianFemale'])

In [8]:
adata.obs.head()

,mammalian1,mammalianlifespan,mammalianfemale
204509080002_R01C02,26.372437,93.886067,0.994351
202897220142_R04C02,1.176586,6.999176,0.991473
204529320092_R01C02,18.776438,73.335119,0.008419
202794570004_R02C01,0.890973,5.332615,0.941965
203531420070_R05C02,10.371315,68.409331,0.009133


After age prediction, the clocks are added to `adata.obs`. Moreover, the percent of missing values for each clock and other metadata are included in `adata.uns`.

In [9]:
adata

AnnData object with n_obs × n_vars = 100 × 37554
    obs: 'mammalian1', 'mammalianlifespan', 'mammalianfemale'
    var: 'percent_na'
    uns: 'mammalian1_percent_na', 'mammalian1_missing_features', 'mammalian1_metadata', 'mammalianlifespan_percent_na', 'mammalianlifespan_missing_features', 'mammalianlifespan_metadata', 'mammalianfemale_percent_na', 'mammalianfemale_missing_features', 'mammalianfemale_metadata'
    layers: 'X_original', None (.X)

#### Mammalian predictors with species declaration

Mammalian2 and mammalian3 types of clocks require species declaration for the reverse transformation of the output into units of years. For the mammalian2 clocks, there are 1756 species in the dictionary with the available variables for reverse transformation; for the mammalian3, there are 1707 species. By default, Homo sapiens is the chosen species.

Let's first have a look at the species that can be used for these clocks by loading the models themselves.

In [10]:
mammalian2_model = pya.pred.load_clock('Mammalian2')
mammalian3_model = pya.pred.load_clock('Mammalian3')

We need to filter the features for the ones that are not CpG sites.

In [11]:
mammalian2_species = [feature for feature in mammalian2_model.features if feature[0:2] != 'cg']
mammalian3_species = [feature for feature in mammalian3_model.features if feature[0:2] != 'cg']
print(f"There are {len(mammalian2_species)} species Latin name features in mammalian2.")
print(f"There are {len(mammalian3_species)} species Latin name features in mammalian3.")

There are 1756 species Latin name features in mammalian2.
There are 1707 species Latin name features in mammalian3.


In [12]:
mammalian2_species[0:10]

['Anaxyrus americanus',
 'Anaxyrus boreas',
 'Anaxyrus canorus',
 'Anaxyrus cognatus',
 'Anaxyrus retiformis',
 'Anaxyrus terrestris',
 'Rhinella marina',
 'Dendrobates auratus',
 'Dendrobates leucomelas',
 'Hyla chrysoscelis']

In [13]:
mammalian3_species[0:10]

['Anaxyrus americanus',
 'Anaxyrus boreas',
 'Anaxyrus canorus',
 'Anaxyrus cognatus',
 'Rhinella marina',
 'Dendrobates auratus',
 'Dendrobates leucomelas',
 'Phyllobates vittatus',
 'Hyla chrysoscelis',
 'Hyla versicolor']

To chose a species, simply add the Latin name as a feature with value 1. In this subset version of the GSE223748 dataset, the species names are not available. Therefore, let's use the naked mole rat (Heterocephalus glaber) as our species. 

Let's first check that it is available in the clocks.

In [14]:
'Heterocephalus glaber' in  mammalian2_species

True

In [15]:
'Heterocephalus glaber' in  mammalian3_species

True

Then, let's add it as a feature to the pandas dataframe and create a new adata object.

In [16]:
df['Heterocephalus glaber'] = 1
adata = pya.pp.df_to_adata(df, imputer_strategy='knn')

Finally, let's make the predictions using the multi-tissue mammalian2 and mammalian3 clocks plus the blood-specific and skin-specific versions.

In [17]:
pya.pred.predict_age(adata, ['Mammalian2', 'MammalianSkin2', 'MammalianBlood2', 'Mammalian3', 'MammalianSkin3', 'MammalianBlood3'])

During age prediction, if the other species are not present in the input data, they will show up as missing features and the value will be automatically replaced with 0. Therefore, those missing features are not necessarily CpG sites. To double check, one can simply go to the adata.uns to check for missing features.

In [18]:
adata.uns['mammalian2_missing_features'][0:10]

['Anaxyrus americanus',
 'Anaxyrus boreas',
 'Anaxyrus canorus',
 'Anaxyrus cognatus',
 'Anaxyrus retiformis',
 'Anaxyrus terrestris',
 'Rhinella marina',
 'Dendrobates auratus',
 'Dendrobates leucomelas',
 'Hyla chrysoscelis']

Finally, let's look at the predictions.

In [19]:
adata.obs.head()

,mammalian2,mammalianskin2,mammalianblood2,mammalian3,mammalianskin3,mammalianblood3
204509080002_R01C02,14.946335,6.562780,15.321810,9.211476,4.328208,13.111774
202897220142_R04C02,17.037182,3.966518,4.293048,17.737573,0.900470,4.240264
204529320092_R01C02,12.065347,13.392643,14.483315,5.950473,7.048246,5.498531
202794570004_R02C01,15.263569,19.451386,7.148743,15.870878,14.483290,11.685419
203531420070_R05C02,6.689490,6.809801,7.602141,4.106029,2.040104,6.293626


For curiosity let's check the correlation between the clocks.

In [20]:
adata.obs.corr('pearson')

,mammalian2,mammalianskin2,mammalianblood2,mammalian3,mammalianskin3,mammalianblood3
mammalian2,1.000000,0.658934,0.609392,0.910119,0.611179,0.717135
mammalianskin2,0.658934,1.000000,0.418290,0.607472,0.900305,0.594863
mammalianblood2,0.609392,0.418290,1.000000,0.463504,0.388197,0.754871
mammalian3,0.910119,0.607472,0.463504,1.000000,0.675022,0.729099
mammalianskin3,0.611179,0.900305,0.388197,0.675022,1.000000,0.646473
mammalianblood3,0.717135,0.594863,0.754871,0.729099,0.646473,1.000000


### Get citation

The doi, citation, and some metadata are automatically added to the AnnData object under `adata.uns[CLOCKNAME_metadata]`.

In [22]:
adata.uns['mammalian2_metadata']

{'clock_name': 'mammalian2',
 'data_type': 'DNA methylation',
 'species': 'multiple species',
 'year': 2023,
 'approved_by_author': '⌛',
 'citation': 'Lu, A. T., et al. "Universal DNA methylation age across mammalian tissues." Nature Aging 3.9 (2023): 1144-1166.',
 'doi': 'https://doi.org/10.1038/s43587-023-00462-6',
 'notes': 'Universal pan-mammalian clock 2 fits relative age (age divided by species maximum lifespan) and the author inverse transformation returns species-adjusted chronological age in years.',
 'research_only': None,
 'tissue': ['multi-tissue'],
 'predicts': ['chronological age'],
 'training_target': ['relative age'],
 'unit': ['years'],
 'model_type': 'elastic net regression',
 'platform': ['Horvath MammalMethylChip40'],
 'population': 'multiple mammalian species',
 'journal': 'Nature Aging',
 'last_author': 'Steve Horvath',
 'n_features': 2572,
 'citations': 390,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'postprocess': 'mammalian2',
 'reference_values': 

<a id="mouse-rrbs"></a>

## Mouse RRBS

This example uses Mus musculus reduced-representation bisulfite sequencing data. We will run RRBS clocks, then map the genomic coordinates to mammalian array probes to run the mammalian clocks.

In [1]:
import pandas as pd
import pyaging as pya
import os
import numpy as np

### Download and load example data

Let's download the publicly available dataset GSE130735 with RRBS samples from mouse. Given it is RRBS, there are millions of CpG sites.

In [2]:
pya.data.download_example_data('GSE130735')

'pyaging_data/GSE130735_subset.pkl'

In [3]:
df = pd.read_pickle('pyaging_data/GSE130735_subset.pkl')

It is important to note that the features for RRBS clocks are the genomic coordinates in the format below.

In [4]:
df.head()

,chr1:3020814,chr1:3020842,chr1:3020877,chr1:3020891,chr1:3020945,chr1:3020971,chr1:3020987,chr1:3021012,chr1:3037802,chr1:3037820,...,chrY:1825397,chrY:4682362,chrY:32122892,chrY:85867071,chrY:85867083,chrY:85867117,chrY:85867137,chrY:85867139,chrY:85867178,chrY:88224179
GSM3752631,0.609,0.25,0.408,0.189,0.068,0.373,0.571,0.252,0.333,0.158,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSM3752625,NaN,NaN,0.973,0.984,0.912,0.915,0.987,0.974,0.991,0.932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSM3752634,NaN,NaN,0.526,0.131,0.000,0.038,0.469,0.769,0.772,0.146,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSM3752620,0.931,0.92,0.988,0.949,0.897,0.921,0.907,0.958,1.000,0.867,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSM3752622,NaN,NaN,0.205,0.382,0.091,0.132,0.174,0.227,0.108,0.053,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Convert data to AnnData object

AnnData objects are highly flexible and are thus our preferred method of organizing data for age prediction.

In [5]:
adata = pya.pp.df_to_adata(df, imputer_strategy='mean') # knn might be a bit slow

This is what the `adata` object looks like:

### Predict age with RRBS clocks

We can either predict one clock at once or all at the same time. For convenience, let's simply input all four available mammalian clocks at once. The function is invariant to the capitalization of the clock name.

In [6]:
pya.pred.predict_age(adata, ['Thompson', 'Meer', 'Petkovich', 'Stubbs'])

Check each clock's output units in `adata.uns['<clock_name>_metadata']['unit']`.

In [7]:
adata.obs.head()

,thompson,meer,petkovich,stubbs
GSM3752631,19.634113,7.315183,8.075177,0.956319
GSM3752625,-1.410461,0.028221,2.953822,-0.074679
GSM3752634,61.058783,21.322178,9.640489,1.389640
GSM3752620,-2.663815,1.611947,3.019351,-0.092847
GSM3752622,20.594114,7.592145,7.104766,0.663506


In [8]:
adata.obs.head()

,thompson,meer,petkovich,stubbs
GSM3752631,19.634113,7.315183,8.075177,0.956319
GSM3752625,-1.410461,0.028221,2.953822,-0.074679
GSM3752634,61.058783,21.322178,9.640489,1.389640
GSM3752620,-2.663815,1.611947,3.019351,-0.092847
GSM3752622,20.594114,7.592145,7.104766,0.663506


After age prediction, the clocks are added to `adata.obs`. Moreover, the percent of missing values for each clock and other metadata are included in `adata.uns`.

In [11]:
adata

AnnData object with n_obs × n_vars = 14 × 1778324
    obs: 'thompson', 'meer', 'petkovich', 'stubbs'
    var: 'percent_na'
    uns: 'imputer_strategy', 'thompson_percent_na', 'thompson_missing_features', 'thompson_metadata', 'meer_percent_na', 'meer_missing_features', 'meer_metadata', 'petkovich_percent_na', 'petkovich_missing_features', 'petkovich_metadata', 'stubbs_percent_na', 'stubbs_missing_features', 'stubbs_metadata'
    layers: 'X_original', None (.X), 'X_imputed'

### Predict age with mammalian clocks

We can predict age by converting the genomic locations directly into the probes from Horvath's methylation array. 

In [12]:
os.system('git clone https://github.com/shorvath/MammalianMethylationConsortium.git')

# Let's read the manifest from the mammalian consortium
annotation_df = pd.read_csv('MammalianMethylationConsortium/Annotations, Amin Haghani/Mammals/Mus_musculus.grcm38.100.HorvathMammalMethylChip40.v1.csv', index_col=0)
annotation_df = annotation_df[~annotation_df.seqnames.isna()]
mm_genomic_locations = 'chr' + annotation_df['seqnames'].astype(str) + ':' + annotation_df['CGstart'].astype(int).astype(str)
mm_genomic_locations = mm_genomic_locations.tolist()
mammalian_probes = annotation_df['CGid'].tolist()
mm_loc_to_probe = dict(zip(mm_genomic_locations, mammalian_probes))

# Let's get the previous RRBS dataset and filter only for the genomic locations in the manifest file
df_columns_set = set(df.columns)
mm_loc_to_probe_set = set(mm_loc_to_probe.keys())
common_columns = df_columns_set.intersection(mm_loc_to_probe_set)
df_converted = df[list(common_columns)].copy()

# Then, convert the genomic location to the probe name
df_converted.columns = [mm_loc_to_probe[col] for col in df_converted.columns]

# Let's clean the GitHub
os.system('rm -r MammalianMethylationConsortium')

Cloning into 'MammalianMethylationConsortium'...


Updating files:  22% (92/405)


Updating files:  24% (101/405)


Updating files:  26% (106/405)


Updating files:  32% (133/405)


...

Updating files:  68% (276/405)
Updating files:  69% (280/405)
Updating files:  70% (284/405)
Updating files:  71% (288/405)
Updating files:  72% (292/405)
Updating files:  73% (296/405)
Updating files:  74% (300/405)
Updating files:  75% (304/405)
Updating files:  76% (308/405)
Updating files:  77% (312/405)
Updating files:  78% (316/405)
Updating files:  79% (320/405)
Updating files:  80% (324/405)
Updating files:  81% (329/405)
Updating files:  82% (333/405)
Updating files:  83% (337/405)
Updating files:  84% (341/405)
Updating files:  85% (345/405)
Updating files:  86% (349/405)
Updating files:  87% (353/405)
Updating files:  88% (357/405)
Updating files:  89% (361/405)
Updating files:  90% (365/405)


Updating files:  92% (373/405)
Updating files:  93% (377/405)


Updating files:  95% (387/405)
Updating files:  96% (389/405)


Updating files:  99% (401/405)
Updating files: 100% (405/405)
Updating files: 100% (405/405), done.


0

In [13]:
df_converted.head()

,cg15915304,cg12160807,cg21973766,cg00409684,cg17404915,cg10437271,cg23653081,cg21474882,cg24818253,cg19044063,...,cg11851831,cg03846926,cg16499073,cg13849080,cg13838363,cg06748976,cg22916990,cg00908709,cg16378867,cg02503970.2
GSM3752631,0.000,0.117,0.211,0.000,0.000,0.000,0.000,0.000,0.082,0.000,...,0.000,0.0,0.353,0.000,0.000,NaN,0.020,0.405,0.008,0.000
GSM3752625,0.000,0.972,0.662,0.000,0.005,0.000,NaN,0.000,0.161,NaN,...,0.113,0.0,0.934,0.037,0.060,NaN,0.000,0.876,0.000,NaN
GSM3752634,0.364,0.986,0.890,0.617,0.068,0.968,NaN,0.014,0.000,NaN,...,0.000,NaN,0.629,0.740,0.000,NaN,0.058,0.971,0.000,NaN
GSM3752620,0.000,1.000,0.816,0.033,0.012,0.009,0.021,0.022,0.347,0.559,...,0.134,NaN,0.875,0.054,0.191,NaN,0.052,0.949,0.000,0.013
GSM3752622,0.000,0.111,0.094,0.000,0.029,0.000,NaN,0.000,0.028,NaN,...,0.062,NaN,0.088,0.000,0.026,NaN,0.000,0.062,0.000,NaN


Now we can finally put the dataframe into pyaging after defining the species as Mus musculus.

In [14]:
df_converted['Mus musculus'] = 1
adata_mammalian = pya.pp.df_to_adata(df_converted, imputer_strategy='mean')

Let's use these five mammalian predictors.

In [15]:
pya.pred.predict_age(adata_mammalian, ['Mammalian1', 'Mammalian2', 'Mammalian3', "MammalianLifespan", "MammalianFemale"])

The models return different quantities. Check each clock's `unit` and `predicts` metadata before comparing predictions.

In [16]:
adata_mammalian.obs

,mammalian1,mammalian2,mammalian3,mammalianlifespan,mammalianfemale
GSM3752631,2.537895,-0.009800,-0.048414,1.202134,0.732238
GSM3752625,3.353935,0.064448,-0.048203,1.480080,0.952105
GSM3752634,4.490610,0.813899,-0.035059,1.400278,0.978554
GSM3752620,3.603802,0.122934,-0.046958,1.644701,0.948646
GSM3752622,2.951263,0.005617,-0.047725,1.385282,0.741336
GSM3752637,5.718515,0.895781,-0.037649,1.399769,0.975114
GSM4558216,7.456245,0.684752,-0.012195,1.448119,0.785830
GSM3752643,5.881943,0.880053,-0.037656,1.408483,0.969979
GSM4558213,6.720080,0.855574,-0.026211,1.477236,0.821308
GSM3752640,6.452934,0.766701,-0.031763,1.371831,0.938162


### Get citation

The doi, citation, and some metadata are automatically added to the AnnData object under `adata.uns[CLOCKNAME_metadata]`.

In [17]:
adata.uns['thompson_metadata']

{'clock_name': 'thompson',
 'data_type': 'DNA methylation',
 'species': 'Mus musculus',
 'year': 2018,
 'approved_by_author': '✅',
 'citation': 'Thompson, M. J., Chwiałkowska, K., Rubbi, L. et al. A multi-tissue full lifespan epigenetic clock for mice. Aging (Albany NY) 10, 2832–2854 (2018).',
 'doi': 'https://doi.org/10.18632/aging.101590',
 'notes': 'Full-lifespan multi-tissue mouse DNA-methylation clock fit by elastic net to RRBS CpG methylation across 1,147 samples from ten tissues and multiple strains; the 582-site all-CpG model estimates chronological age and detects intervention- and genotype-associated age acceleration.',
 'research_only': None,
 'tissue': ['adipose tissue',
  'blood',
  'cerebellum',
  'brain cortex',
  'heart',
  'kidney',
  'liver',
  'lung',
  'skeletal muscle',
  'spleen'],
 'predicts': ['chronological age'],
 'training_target': ['chronological age'],
 'unit': ['months'],
 'model_type': 'elastic net regression',
 'platform': ['RRBS'],
 'population': 'mice'

In [18]:
adata.uns['meer_metadata']

{'clock_name': 'meer',
 'data_type': 'DNA methylation',
 'species': 'Mus musculus',
 'year': 2018,
 'approved_by_author': '⌛',
 'citation': 'Meer, M. V., Podolskiy, D. I., Tyshkovskiy, A. & Gladyshev, V. N. A whole lifespan mouse multi-tissue DNA methylation clock. eLife 7, e40675 (2018).',
 'doi': 'https://doi.org/10.7554/elife.40675',
 'notes': 'Whole Lifespan Multi-Tissue (WLMT) mouse clock trained by elastic net on RRBS methylation percentages from untreated wild-type C57BL/6 samples.',
 'research_only': None,
 'tissue': ['multi-tissue'],
 'predicts': ['chronological age'],
 'training_target': ['chronological age'],
 'unit': ['days'],
 'model_type': 'elastic net regression',
 'platform': ['RRBS'],
 'population': 'mice',
 'journal': 'eLife',
 'last_author': 'Vadim N. Gladyshev',
 'n_features': 435,
 'citations': 203,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0'}

In [19]:
adata.uns['petkovich_metadata']

{'clock_name': 'petkovich',
 'data_type': 'DNA methylation',
 'species': 'Mus musculus',
 'year': 2017,
 'approved_by_author': '⌛',
 'citation': 'Petkovich, Daniel A., et al. "Using DNA methylation profiling to evaluate biological age and longevity interventions." Cell metabolism 25.4 (2017): 954-960.',
 'doi': 'https://doi.org/10.1016/j.cmet.2017.03.016',
 'notes': 'Mouse blood DNA-methylation age clock built by regression on reduced-representation bisulfite-sequencing CpGs, estimating biological age and shown to be slowed by lifespan-extending interventions such as caloric restriction and dwarfism.',
 'research_only': None,
 'tissue': ['whole blood'],
 'predicts': ['chronological age'],
 'training_target': ['chronological age'],
 'unit': ['months'],
 'model_type': 'elastic net regression',
 'platform': ['bisulfite sequencing'],
 'population': 'mice',
 'journal': 'Cell Metabolism',
 'last_author': 'Vadim N. Gladyshev',
 'n_features': 90,
 'citations': 441,
 'citations_date': '2026-07-

In [20]:
adata.uns['stubbs_metadata']

{'clock_name': 'stubbs',
 'data_type': 'DNA methylation',
 'species': 'Mus musculus',
 'year': 2017,
 'approved_by_author': '⌛',
 'citation': 'Stubbs, T. M., et al. “Multi-tissue DNA methylation age predictor in mouse.” Genome Biology 18: 68 (2017).',
 'doi': 'https://doi.org/10.1186/s13059-017-1203-5',
 'notes': 'Mouse multi-tissue RRBS age predictor: 17,992 common input loci are normalized and reduced to 329 nonzero clock sites, then quadratically calibrated.',
 'research_only': None,
 'tissue': ['multi-tissue',
  'liver',
  'lung',
  'heart',
  'brain cortex',
  'skeletal muscle',
  'cerebellum',
  'spleen'],
 'predicts': ['chronological age'],
 'training_target': ['chronological age'],
 'unit': ['months'],
 'model_type': 'quadratically calibrated elastic net regression',
 'platform': ['RRBS'],
 'population': 'mice',
 'journal': 'Genome Biology',
 'last_author': 'Wolf Reik',
 'n_features': 17992,
 'citations': 430,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'preprocess'

In [21]:
adata_mammalian.uns['mammalian1_metadata']

{'clock_name': 'mammalian1',
 'data_type': 'DNA methylation',
 'species': 'multiple species',
 'year': 2023,
 'approved_by_author': '⌛',
 'citation': 'Lu, A. T., et al. "Universal DNA methylation age across mammalian tissues." Nature Aging 3.9 (2023): 1144-1166.',
 'doi': 'https://doi.org/10.1038/s43587-023-00462-6',
 'notes': 'Universal pan-mammalian clock 1: elastic-net regression of log-transformed chronological age on conserved mammalian-array CpGs, back-transformed to years.',
 'research_only': None,
 'tissue': ['multi-tissue'],
 'predicts': ['chronological age'],
 'training_target': ['chronological age'],
 'unit': ['years'],
 'model_type': 'elastic net regression',
 'platform': ['Horvath MammalMethylChip40'],
 'population': 'multiple mammalian species',
 'journal': 'Nature Aging',
 'last_author': 'Steve Horvath',
 'n_features': 335,
 'citations': 390,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'postprocess': 'anti_logp2'}

In [22]:
adata_mammalian.uns['mammalianlifespan_metadata']

{'clock_name': 'mammalianlifespan',
 'data_type': 'DNA methylation',
 'species': 'multiple species',
 'year': 2024,
 'approved_by_author': '⌛',
 'citation': 'Li, Caesar Z., et al. "Epigenetic predictors of species maximum life span and other life-history traits in mammals." Science Advances 10.23 (2024): eadm7273.',
 'doi': 'https://doi.org/10.1126/sciadv.adm7273',
 'notes': 'Pan-mammalian tissue-agnostic elastic-net predictor fitted to log species maximum life span from conserved CpG methylation; pyaging exponentiates the linear output to years.',
 'research_only': None,
 'tissue': ['multi-tissue'],
 'predicts': ['species maximum lifespan'],
 'training_target': ['species maximum lifespan'],
 'unit': ['years'],
 'model_type': 'elastic net regression',
 'platform': ['Horvath MammalMethylChip40'],
 'population': 'multiple mammalian species',
 'journal': 'Science Advances',
 'last_author': 'Steve Horvath',
 'n_features': 152,
 'citations': 5,
 'citations_date': '2026-07-05',
 'version': '